# LOB Visualization: Aggressive Scenario

Интерактивная визуализация эволюции книги заявок для aggressive scenario.
- Визуализация **COND + GEN** (с aggressive order insertions)

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

## Data Loading

In [2]:
# Only set the path to data folder - everything else is auto-discovered
DATA_PATH = Path("/app/output/evalsequences/aggressive_scenario/exp_36_20260205_174137_gen_i5_c30_btw10_sell")

# Samples to exclude from analysis (list of sample_ids)
EXCLUDE_SAMPLES = [266]
# EXCLUDE_SAMPLES = [4, 5, 1894]  # Example: exclude specific sample IDs

# 19667

import re

def discover_data_params(data_path):
    """Auto-discover ticker and sample info from files in data_cond folder.
    
    Expects files like: GOOG_2023-01-13_orderbook_real_id_1555.csv
    Returns: (ticker, sample_info) where sample_info is dict {sample_id: date}
    """
    cond_dir = data_path / "data_cond"
    pattern = re.compile(r"^(.+?)_(\d{4}-\d{2}-\d{2})_orderbook_real_id_(\d+)\.csv$")
    
    tickers = set()
    sample_info = {}  # {sample_id: date}
    
    for f in cond_dir.glob("*_orderbook_real_id_*.csv"):
        match = pattern.match(f.name)
        if match:
            tickers.add(match.group(1))
            sample_id = int(match.group(3))
            date = match.group(2)
            sample_info[sample_id] = date
    
    if not sample_info:
        raise ValueError(f"No orderbook files found in {cond_dir}")
    if len(tickers) > 1:
        raise ValueError(f"Multiple tickers found: {tickers}")
    
    ticker = tickers.pop()
    dates = set(sample_info.values())
    
    print(f"Discovered: ticker={ticker}, dates={sorted(dates)}, n_samples={len(sample_info)}")
    return ticker, sample_info

def load_all_data():
    """Load and concatenate all data. Returns cond_lens for junction marking.
    
    Note: For aggressive scenario, we only have COND and GEN data (no REAL).
    """
    ticker, sample_info = discover_data_params(DATA_PATH)
    
    gen_books = {}
    gen_msgs = {}
    cond_lens = {}  # Store conditioning sequence length for each sample
    
    for sid, date in sorted(sample_info.items()):
        # Skip excluded samples
        if sid in EXCLUDE_SAMPLES:
            continue
            
        # Load conditioning data
        cond_book = np.loadtxt(DATA_PATH / f"data_cond/{ticker}_{date}_orderbook_real_id_{sid}.csv", delimiter=',')
        cond_msg = np.loadtxt(DATA_PATH / f"data_cond/{ticker}_{date}_message_real_id_{sid}.csv", delimiter=',')
        
        # Load generated continuation (with aggressive orders)
        gen_book = np.loadtxt(DATA_PATH / f"data_gen/{ticker}_{date}_orderbook_real_id_{sid}_gen_id_0.csv", delimiter=',')
        gen_msg = np.loadtxt(DATA_PATH / f"data_gen/{ticker}_{date}_message_real_id_{sid}_gen_id_0.csv", delimiter=',')
        
        # Store conditioning length (junction point)
        cond_lens[sid] = cond_book.shape[0]
        
        # Concatenate COND + GEN
        gen_books[sid] = np.vstack([cond_book, gen_book])
        gen_msgs[sid] = np.vstack([cond_msg, gen_msg])
    
    excluded_count = len([s for s in sample_info.keys() if s in EXCLUDE_SAMPLES])
    print(f"Loaded {len(gen_books)} samples (excluded {excluded_count} samples)")
    if excluded_count > 0:
        print(f"Excluded samples: {EXCLUDE_SAMPLES}")
    return gen_books, gen_msgs, cond_lens

gen_books, gen_msgs, cond_lens = load_all_data()

Discovered: ticker=GOOG, dates=['2023-01-03', '2023-01-04', '2023-01-05', '2023-01-06', '2023-01-09', '2023-01-10', '2023-01-11', '2023-01-12', '2023-01-13'], n_samples=2048
Loaded 2047 samples (excluded 1 samples)
Excluded samples: [266]


## Visualization Function

In [ ]:
AGGRESSIVE_ORDER_ID = 77777777  # Marker for aggressive orders
SENTINEL_VALUE = -2147483647  # Sentinel for missing book levels
MAX_QTY_DISPLAY = 5000  # Max quantity to display (clips extremes)

def sanitize_book_row(book_row):
    """Replace sentinel values with 0 for proper handling."""
    result = book_row.copy().astype(float)
    result[np.abs(result) > 1e9] = 0  # Filter extreme values
    return result

def extract_quantities(book_row):
    """Extract quantities from orderbook row (LOBSTER interleaved format).
    Format: [ask_price1, ask_qty1, bid_price1, bid_qty1, ask_price2, ...]
    Returns: [bid_qty1, ..., bid_qty10, ask_qty1, ..., ask_qty10]
    """
    row = sanitize_book_row(book_row)
    ask_qtys = row[1::4]   # indices 1, 5, 9, ... (10 values)
    bid_qtys = row[3::4]   # indices 3, 7, 11, ... (10 values)
    return np.concatenate([bid_qtys, ask_qtys])

def compute_queued_volumes(book_array):
    """Compute bid/ask/total queued volume over time."""
    T = book_array.shape[0]
    ask_vol = np.zeros(T)
    bid_vol = np.zeros(T)
    for t in range(T):
        row = sanitize_book_row(book_array[t])
        ask_vol[t] = np.sum(np.abs(row[1::4]))
        bid_vol[t] = np.sum(np.abs(row[3::4]))
    return bid_vol, ask_vol, bid_vol + ask_vol

def interactive_lob_plot(books, msgs, cond_lens, title="LOB", data_type="GEN"):
    """Interactive LOB visualization for aggressive scenario."""
    
    # Controls
    id_dd = widgets.Dropdown(options=sorted(books.keys()), description="Sample ID:")
    time_slider = widgets.IntSlider(min=1, max=1, step=1, description="t:")
    btn_prev = widgets.Button(description="←", layout=widgets.Layout(width="50px"))
    btn_next = widgets.Button(description="→", layout=widgets.Layout(width="50px"))
    btn_junction = widgets.Button(description="→ Junction", layout=widgets.Layout(width="100px"))
    btn_next_aggr = widgets.Button(description="→ Next Aggr", layout=widgets.Layout(width="100px"))
    
    # Display boxes
    sample_box = widgets.HTML()
    msg_box = widgets.HTML()
    vol_box = widgets.HTML()
    
    # Figure: book panels + volumes
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[f"{title}: Book t-1", f"{title}: Book t", f"{title}: Volumes"],
        column_widths=[0.3, 0.3, 0.4]
    )
    fig.add_trace(go.Bar(x=[], y=[], name="t-1"), row=1, col=1)
    fig.add_trace(go.Bar(x=[], y=[], name="t"), row=1, col=2)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Bid'), row=1, col=3)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Ask'), row=1, col=3)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Total'), row=1, col=3)
    fig.add_shape(type="line", x0=0, x1=0, y0=0, y1=1, xref="x3", yref="paper",
                  line=dict(color="gray", width=1, dash="dash"))
    fig.add_shape(type="line", x0=0, x1=0, y0=0, y1=1, xref="x3", yref="paper",
                  line=dict(color="red", width=3))
    
    # Fixed axis ranges for book plots
    fig.update_layout(
        width=1200, height=400, showlegend=False, template='plotly_white',
        margin=dict(l=30, r=30, t=50, b=30),
        xaxis=dict(range=[-11, 10]),   # Level indices
        xaxis2=dict(range=[-11, 10]),
        yaxis=dict(range=[-MAX_QTY_DISPLAY, MAX_QTY_DISPLAY]),   # Fixed qty range
        yaxis2=dict(range=[-MAX_QTY_DISPLAY, MAX_QTY_DISPLAY]),
    )
    fig_widget = go.FigureWidget(fig)
    
    _vol_cache = {}
    _aggr_indices_cache = {}
    
    def find_aggressive_indices(msg_arr):
        return np.where(msg_arr[:, 2].astype(int) == AGGRESSIVE_ORDER_ID)[0]
    
    def format_message(msg_row):
        m = msg_row.astype(int)
        order_id, et, sz, price, dr = m[2], m[1], m[3], m[4], m[5]
        et_map = {1: "Limit", 2: "PartialCancel", 3: "Delete", 4: "Execution"}
        dr_map = {1: "Buy", -1: "Sell"}
        info = f"{et_map.get(et, '?')} | {dr_map.get(dr, '?')} | size={sz} | ${price/100:.2f}"
        if order_id == AGGRESSIVE_ORDER_ID:
            return f"<span style='color:red; font-weight:bold'>⚡ AGGRESSIVE ⚡</span> {info}"
        return f"{info} | oid={order_id}"
    
    def update_sample(*_):
        sid = id_dd.value
        time_slider.min = 1
        time_slider.max = books[sid].shape[0] - 1
        time_slider.value = 1
        
        if sid not in _vol_cache:
            _vol_cache[sid] = compute_queued_volumes(books[sid])
        if sid not in _aggr_indices_cache:
            _aggr_indices_cache[sid] = find_aggressive_indices(msgs[sid])
        
        bid_v, ask_v, tot_v = _vol_cache[sid]
        xs = np.arange(len(bid_v))
        junction = cond_lens[sid]
        
        with fig_widget.batch_update():
            fig_widget.data[2].x = xs
            fig_widget.data[2].y = bid_v
            fig_widget.data[3].x = xs
            fig_widget.data[3].y = ask_v
            fig_widget.data[4].x = xs
            fig_widget.data[4].y = tot_v
            fig_widget.layout.shapes[1].x0 = junction
            fig_widget.layout.shapes[1].x1 = junction
        update_plot()
    
    def update_plot(*_):
        sid = id_dd.value
        t = time_slider.value
        arr = books[sid]
        msg_arr = msgs[sid]
        junction = cond_lens[sid]
        
        q0 = extract_quantities(arr[t - 1])
        q1 = extract_quantities(arr[t])
        
        # Clip to max display range
        q0 = np.clip(q0, -MAX_QTY_DISPLAY, MAX_QTY_DISPLAY)
        q1 = np.clip(q1, -MAX_QTY_DISPLAY, MAX_QTY_DISPLAY)
        
        q0_signed = np.concatenate([-np.abs(q0[:10]), np.abs(q0[10:])])
        q1_signed = np.concatenate([-np.abs(q1[:10]), np.abs(q1[10:])])
        
        diff = np.abs(q1) - np.abs(q0)
        x = np.arange(len(q0)) - len(q0) // 2
        colors = ['gray' if abs(d) < 1e-8 else ('red' if d > 0 else 'blue') for d in diff]
        
        # Get prices
        row_t = sanitize_book_row(arr[t])
        best_ask = row_t[0] if row_t[0] > 0 else 0
        best_bid = row_t[2] if row_t[2] > 0 else 0
        mid_price = (best_ask + best_bid) / 2 if best_ask > 0 and best_bid > 0 else 0
        
        with fig_widget.batch_update():
            fig_widget.data[0].x = x
            fig_widget.data[0].y = q0_signed
            fig_widget.data[0].marker.color = 'gray'
            fig_widget.data[1].x = x
            fig_widget.data[1].y = q1_signed
            fig_widget.data[1].marker.color = colors
            fig_widget.layout.shapes[0].x0 = t
            fig_widget.layout.shapes[0].x1 = t
            fig_widget.layout.annotations[0].text = f"{title}: Book t={t-1}"
            fig_widget.layout.annotations[1].text = f"{title}: Book t={t}"
        
        sample_type = "COND" if t < junction else data_type
        sample_box.value = f"<b>{data_type}</b> | GEN | junction@{junction} | <b>Mid=${mid_price/100:.2f}</b> | Bid1=${best_bid/100:.2f} Ask1=${best_ask/100:.2f}"
        
        msg_idx = t - 1
        if 0 <= msg_idx < len(msg_arr):
            msg_box.value = format_message(msg_arr[msg_idx])
        
        bid_v, ask_v, tot_v = _vol_cache[sid]
        vol_box.value = f"Vol: Bid={bid_v[t]:.0f} Ask={ask_v[t]:.0f} Total={tot_v[t]:.0f}"
    
    def on_prev(_):
        if time_slider.value > time_slider.min:
            time_slider.value -= 1
    
    def on_next(_):
        if time_slider.value < time_slider.max:
            time_slider.value += 1
    
    def on_junction(_):
        sid = id_dd.value
        junction = cond_lens[sid]
        if time_slider.min <= junction <= time_slider.max:
            time_slider.value = junction
    
    def on_next_aggressive(_):
        sid = id_dd.value
        aggr_indices = _aggr_indices_cache.get(sid, [])
        current_t = time_slider.value
        for idx in aggr_indices:
            if idx + 1 > current_t:
                time_slider.value = idx + 1
                return
        if len(aggr_indices) > 0:
            time_slider.value = aggr_indices[0] + 1
    
    id_dd.observe(lambda _: update_sample(), names='value')
    time_slider.observe(lambda _: update_plot(), names='value')
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_junction.on_click(on_junction)
    btn_next_aggr.on_click(on_next_aggressive)
    
    update_sample()
    
    controls = widgets.HBox([id_dd, btn_prev, btn_next, btn_junction, btn_next_aggr, time_slider])
    display(widgets.HTML(f"<h3>{title}</h3>"))
    display(controls)
    display(fig_widget)
    display(sample_box)
    display(msg_box)
    display(vol_box)

## COND + GEN (Aggressive Scenario) / /app/lob_impact/1.aggressive_scenario_s5_rnd.py

In [11]:
interactive_lob_plot(gen_books, gen_msgs, cond_lens, title="COND + GEN (Aggressive)", data_type="GEN")

HTML(value='<h3>COND + GEN (Aggressive)</h3>')

FigureWidget({
    'data': [{'marker': {'color': 'gray'},
              'name': 't-1',
              'type': 'bar',
              'uid': '50eb47f2-2801-4f82-b6f2-ea3ecb177bd3',
              'x': array([-10,  -9,  -8,  -7,  -6,  -5,  -4,  -3,  -2,  -1,   0,   1,   2,   3,
                            4,   5,   6,   7,   8,   9]),
              'xaxis': 'x',
              'y': array([ -20., -143., -212., -150., -150., -125., -275., -100., -125., -100.,
                            89.,   10.,   20.,  100.,   51.,   25.,  600.,    1.,   25.,  200.]),
              'yaxis': 'y'},
             {'marker': {'color': [red, red, blue, gray, blue, red, blue, red,
                                   blue, red, gray, gray, gray, gray, gray, gray,
                                   gray, gray, gray, gray]},
              'name': 't',
              'type': 'bar',
              'uid': '31fd7736-7fd9-424b-a29d-cbc53cc4cc4d',
              'x': array([-10,  -9,  -8,  -7,  -6,  -5,  -4,  -3,  -2,  -1,   0

HTML(value='')

HTML(value='Sample ID: 4 | Segment: <b>COND</b> | junction @ 501')

HTML(value='<b>Execution • Buy • size=20 • price=900900</b> | order_id=28408858<br>raw: [34205, 4, 28408858, 2…

HTML(value='<b>Volume:</b> Bid=1580 • Ask=1121 • Total=2701')

## Aggregated Midprice Return

In [ ]:
def compute_midprice(book_array):
    """Compute midprice: (best_ask + best_bid) / 2.
    LOBSTER format: Index 0 = best ask price, Index 2 = best bid price
    Handles sentinel values (-2147483647) by treating them as NaN.
    """
    ask_prices = book_array[:, 0].astype(float)
    bid_prices = book_array[:, 2].astype(float)
    # Replace sentinel values with NaN
    ask_prices[ask_prices == SENTINEL_VALUE] = np.nan
    bid_prices[bid_prices == SENTINEL_VALUE] = np.nan
    return (ask_prices + bid_prices) / 2

def plot_aggregated_midprice_return(books, msgs, cond_lens, title="Midprice Mean ±1 Std"):
    """Plot aggregated midprice return across all samples with aggressive order markers.
    
    Args:
        books: dict {sample_id: book_array}
        msgs: dict {sample_id: msg_array}
        cond_lens: dict {sample_id: conditioning_length} for junction marking
        title: plot title
    """
    # Compute midprice return for each sample
    midprice_returns = []
    min_len = min(b.shape[0] for b in books.values())
    
    for sid, book_array in books.items():
        midprice = compute_midprice(book_array[:min_len])
        # Skip samples with NaN midprices
        if np.any(np.isnan(midprice)):
            continue
        midprice_return = midprice - midprice[0]  # subtract first value
        midprice_returns.append(midprice_return)
    
    if not midprice_returns:
        print("No valid samples (all have NaN midprices)")
        return
    
    # Stack and compute mean/std
    all_returns = np.stack(midprice_returns, axis=0)  # shape: (n_samples, T)
    mean_return = np.mean(all_returns, axis=0)
    std_return = np.std(all_returns, axis=0)
    
    steps = np.arange(min_len)
    junction = list(cond_lens.values())[0]  # assume same for all samples
    
    # Find all aggressive order indices across all samples
    all_aggr_indices = set()
    for sid, msg_arr in msgs.items():
        aggr_idx = np.where(msg_arr[:, 2].astype(int) == AGGRESSIVE_ORDER_ID)[0]
        # Message at index i corresponds to book state at index i+1
        for idx in aggr_idx:
            if idx + 1 < min_len:
                all_aggr_indices.add(idx + 1)
    all_aggr_indices = sorted(all_aggr_indices)
    
    # Create figure
    fig = go.Figure()
    
    # Add ±1 std shaded area
    fig.add_trace(go.Scatter(
        x=np.concatenate([steps, steps[::-1]]),
        y=np.concatenate([mean_return + std_return, (mean_return - std_return)[::-1]]),
        fill='toself',
        fillcolor='rgba(255, 99, 71, 0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        name='±1 Std',
        showlegend=False
    ))
    
    # Add mean line
    fig.add_trace(go.Scatter(
        x=steps,
        y=mean_return,
        mode='lines',
        name='Mean',
        line=dict(color='red', width=2)
    ))
    
    # Add aggressive order markers as vertical lines
    for i, aggr_t in enumerate(all_aggr_indices):
        fig.add_vline(
            x=aggr_t, 
            line_color="green", 
            line_width=1.5, 
            line_dash="solid",
            opacity=0.7,
            annotation_text="" if i > 0 else "Aggr",
            annotation_position="top"
        )
    
    # Add markers on mean line at aggressive order points
    if all_aggr_indices:
        aggr_y = [mean_return[t] for t in all_aggr_indices]
        fig.add_trace(go.Scatter(
            x=all_aggr_indices,
            y=aggr_y,
            mode='markers',
            name='Aggressive Orders',
            marker=dict(color='green', size=10, symbol='triangle-up')
        ))
    
    # Add zero reference line (dashed gray)
    fig.add_hline(y=0, line_dash="dash", line_color="gray")
    
    # Add junction line (vertical)
    fig.add_vline(x=junction, line_color="gray", line_width=1, opacity=0.5,
                  annotation_text="Junction", annotation_position="top")
    
    fig.update_layout(
        title=title,
        xaxis_title='Steps (sampled midprice points)',
        yaxis_title='Price - first price',
        width=1000,
        height=600,
        template='plotly_white',
        legend=dict(x=1, y=1, xanchor='right')
    )
    
    fig.show()
    
    # Print statistics
    print(f"Samples: {len(midprice_returns)}, Steps: {min_len}, Junction: {junction}")
    print(f"Final mean return: {mean_return[-1]:.2f} ± {std_return[-1]:.2f}")
    print(f"Aggressive orders at steps: {all_aggr_indices}")

plot_aggregated_midprice_return(gen_books, gen_msgs, cond_lens)

## Most Volatile Samples

Список самых волатильных сэмплов для детального изучения. Скопируй sample_id и вставь в dropdown выше или добавь в `EXCLUDE_SAMPLES` в начале ноутбука.

In [6]:
def compute_volatility_stats_gen(books, cond_lens, top_n=30):
    """
    Compute volatility statistics for each sample (GEN only).
    
    Metrics:
    - gen_range: max - min midprice in GEN continuation
    - gen_std: std of midprice returns in GEN
    - gen_return: total return from junction to end
    
    Returns DataFrame sorted by gen_range (most volatile first).
    """
    stats = []
    
    for sid in books.keys():
        junction = cond_lens[sid]
        
        gen_mid = compute_midprice(books[sid])
        
        # Continuation part only (after junction)
        gen_cont = gen_mid[junction:]
        
        # Range (max - min) - measure of price movement
        gen_range = gen_cont.max() - gen_cont.min()
        
        # Std of returns
        gen_returns = np.diff(gen_cont)
        gen_std = np.std(gen_returns) if len(gen_returns) > 0 else 0
        
        # Total return (final - junction price)
        gen_total_return = gen_mid[-1] - gen_mid[junction]
        
        # Midprice at start and end
        start_price = gen_mid[junction]
        end_price = gen_mid[-1]
        
        stats.append({
            'sample_id': sid,
            'gen_range': gen_range,
            'gen_std': gen_std,
            'gen_return': gen_total_return,
            'start_price': start_price,
            'end_price': end_price,
        })
    
    df = pd.DataFrame(stats)
    return df

# Compute stats
vol_df = compute_volatility_stats_gen(gen_books, cond_lens)

print("=" * 80)
print("TOP 30 MOST VOLATILE SAMPLES (by GEN midprice range)")
print("=" * 80)
top_range = vol_df.nlargest(30, 'gen_range')[['sample_id', 'gen_range', 'gen_return', 'gen_std', 'start_price', 'end_price']]
print(top_range.to_string(index=False))

print("\n" + "=" * 80)
print("TOP 30 LARGEST POSITIVE RETURNS (price went up)")
print("=" * 80)
top_pos = vol_df.nlargest(30, 'gen_return')[['sample_id', 'gen_return', 'gen_range', 'gen_std', 'start_price', 'end_price']]
print(top_pos.to_string(index=False))

print("\n" + "=" * 80)
print("TOP 30 LARGEST NEGATIVE RETURNS (price went down)")
print("=" * 80)
top_neg = vol_df.nsmallest(30, 'gen_return')[['sample_id', 'gen_return', 'gen_range', 'gen_std', 'start_price', 'end_price']]
print(top_neg.to_string(index=False))

TOP 30 MOST VOLATILE SAMPLES (by GEN midprice range)
 sample_id  gen_range  gen_return   gen_std  start_price  end_price
     19667     1150.0      1100.0 21.544120     915200.0   916300.0
      5250     1000.0      -800.0 21.646519     879950.0   879150.0
      7453     1000.0     -1000.0 24.879346     866450.0   865450.0
      1996      950.0       550.0 26.738045     898200.0   898750.0
     12360      950.0      -950.0 11.799872     884100.0   883150.0
     16673      950.0      -950.0 16.688701     922200.0   921250.0
         5      900.0      -900.0 24.627799     901650.0   900750.0
        68      900.0      -900.0 14.715854     903900.0   903000.0
      7452      850.0      -700.0 15.263251     866800.0   866100.0
     19697      850.0       750.0 18.729354     916400.0   917150.0
      7839      800.0       650.0 17.609437     862200.0   862850.0
       279      750.0      -700.0 15.263251     910050.0   909350.0
      1955      750.0       100.0 39.317724     911450.0   9115

In [7]:
# Find outliers - samples with extreme GEN returns
OUTLIER_THRESHOLD = 1000  # Threshold for |gen_return|

print("=" * 80)
print(f"OUTLIERS: Samples with extreme GEN midprice (|gen_return| > {OUTLIER_THRESHOLD})")
print("=" * 80)

outliers = vol_df[vol_df['gen_return'].abs() > OUTLIER_THRESHOLD].sort_values('gen_return', key=abs, ascending=False)
print(f"Found {len(outliers)} outliers:\n")
print(outliers[['sample_id', 'gen_return', 'gen_range', 'gen_std', 'start_price', 'end_price']].to_string(index=False))

print("\n" + "=" * 80)
print("Outlier sample_ids (copy-paste to EXCLUDE_SAMPLES):")
print("=" * 80)
outlier_ids = outliers['sample_id'].tolist()
print(f"EXCLUDE_SAMPLES = {outlier_ids}")

print("\nDetailed list:")
for _, row in outliers.iterrows():
    print(f"  {row['sample_id']:>6}  →  GEN return: {row['gen_return']:>8.0f}  |  range: {row['gen_range']:>6.0f}")

OUTLIERS: Samples with extreme GEN midprice (|gen_return| > 1000)
Found 1 outliers:

 sample_id  gen_return  gen_range  gen_std  start_price  end_price
     19667      1100.0     1150.0 21.54412     915200.0   916300.0

Outlier sample_ids (copy-paste to EXCLUDE_SAMPLES):
EXCLUDE_SAMPLES = [19667]

Detailed list:
  19667.0  →  GEN return:     1100  |  range:   1150


## Market Impact Beta Calculation

In [8]:
# === Market Impact Beta Calculation and Visualization ===

def compute_market_impact_beta(books, msgs, cond_lens, iteration_filter=None, title_suffix=""):
    """
    Compute and visualize market impact beta using OLS regression.
    
    Args:
        iteration_filter: None = all iterations, or int = only that iteration
        title_suffix: additional text for title
    """
    eps = 1e-12

    points = []  # (x, y, sample_id, iteration)
    alphas = []
    
    for sid in books.keys():
        msg_arr = msgs[sid]
        book_arr = books[sid]
        junction = cond_lens[sid]

        aggr_mask = msg_arr[:, 2].astype(int) == AGGRESSIVE_ORDER_ID
        aggr_indices = np.where(aggr_mask)[0]
        
        if len(aggr_indices) < 2:
            continue

        sizes = msg_arr[aggr_indices, 3].astype(float)
        prices = msg_arr[aggr_indices, 4].astype(float)

        first_idx = aggr_indices[0]
        ref_price = (book_arr[first_idx, 0] + book_arr[first_idx, 2]) / 2

        if ref_price <= 0:
            continue

        Q_cum = np.cumsum(sizes)
        notional_cum = np.cumsum(sizes * prices)
        vwap = notional_cum / np.maximum(Q_cum, eps)
        impact = np.abs(vwap - ref_price) / ref_price

        gen_msgs = msg_arr[junction:]
        exec_mask = gen_msgs[:, 1].astype(int) == 4
        V_exp = np.sum(gen_msgs[exec_mask, 3].astype(float)) if np.any(exec_mask) else 1.0
        V_exp = max(V_exp, eps)

        midprices = (book_arr[junction:, 0] + book_arr[junction:, 2]) / 2
        H, L = np.max(midprices), np.min(midprices)
        
        if H > L and L > 0:
            eta = np.log(H / L) / 0.8325546
            alphas.append(np.log(max(eta, eps)))

        valid_mask = impact > eps
        if np.sum(valid_mask) < 2:
            continue

        x_vals = np.log(Q_cum[valid_mask] / V_exp)
        y_vals = np.log(impact[valid_mask])
        iterations = np.arange(1, len(sizes) + 1)[valid_mask]
        
        for x, y, it in zip(x_vals, y_vals, iterations):
            points.append((x, y, sid, int(it)))

    if not points or not alphas:
        print("No valid data for beta estimation")
        return

    # Convert and filter
    all_X = np.array([p[0] for p in points])
    all_Y = np.array([p[1] for p in points])
    all_sids = np.array([p[2] for p in points])
    all_iters = np.array([p[3] for p in points])
    
    if iteration_filter is not None:
        mask = all_iters == iteration_filter
        X = all_X[mask]
        Y = all_Y[mask]
        sids = all_sids[mask]
        iters = all_iters[mask]
    else:
        X, Y, sids, iters = all_X, all_Y, all_sids, all_iters
    
    alpha_global = np.mean(alphas)

    y_adj = Y - alpha_global
    valid = np.isfinite(X) & np.isfinite(y_adj) & (X != 0)
    
    if np.sum(valid) < 2:
        print("Not enough valid points")
        return
        
    beta_ols = float(np.dot(X[valid], y_adj[valid]) / np.dot(X[valid], X[valid]))

    # Group by sample
    from collections import defaultdict
    sample_points = defaultdict(list)
    for x, y, sid, it in zip(X, Y, sids, iters):
        sample_points[sid].append((x, y, it))

    fig = go.Figure()

    import plotly.express as px
    colors = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
    
    for i, (sid, pts) in enumerate(sorted(sample_points.items())):
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        its = [p[2] for p in pts]
        hover = [f"Sample: {sid}<br>Iter: {it}<br>x: {x:.3f}<br>y: {y:.3f}" 
                 for x, y, it in zip(xs, ys, its)]
        
        fig.add_trace(go.Scatter(
            x=xs, y=ys, mode='markers',
            marker=dict(size=9, opacity=0.8, color=colors[i % len(colors)]),
            text=hover, hoverinfo='text',
            name=f'Sample {sid}',
            legendgroup=str(sid)
        ))

    x_range = np.linspace(X.min(), X.max(), 100)
    fig.add_trace(go.Scatter(
        x=x_range, y=alpha_global + beta_ols * x_range,
        mode='lines', line=dict(color='red', width=3),
        name=f'OLS: β = {beta_ols:.3f}'
    ))

    fig.add_trace(go.Scatter(
        x=x_range, y=alpha_global + 0.5 * x_range,
        mode='lines', line=dict(color='black', width=3, dash='dash'),
        name='Theory: β = 0.5'
    ))

    iter_text = f"iter={iteration_filter}" if iteration_filter else "all iterations"
    fig.update_layout(
        title=f'Market Impact: β_OLS = {beta_ols:.4f} ({iter_text}){title_suffix}',
        xaxis_title='log(Q / V_exp)',
        yaxis_title='log(Impact)',
        width=1100, height=600,
        template='plotly_white',
        hovermode='closest',
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.02, font=dict(size=10))
    )
    fig.show()

    print(f"Points: {len(X)}, Alpha: {alpha_global:.4f}, Beta: {beta_ols:.4f}")
    return beta_ols

# 1. All iterations
print("=== All iterations ===")
beta_all = compute_market_impact_beta(gen_books, gen_msgs, cond_lens)

# 2. Only last iteration (5)
print("\n=== Only last iteration (5) ===")
beta_last = compute_market_impact_beta(gen_books, gen_msgs, cond_lens, iteration_filter=5)

=== All iterations ===
No valid data for beta estimation

=== Only last iteration (5) ===
No valid data for beta estimation


In [9]:
# === Beta Evolution: β(a) - beta starting from iteration a ===

def compute_beta_evolution(books, msgs, cond_lens):
    """
    Compute beta for different starting points a = 1, 2, 3, ...
    Shows how beta changes when excluding early iterations.
    """
    eps = 1e-12
    
    # Collect all points with iteration info
    points = []  # (x, y, iteration)
    alphas = []
    
    for sid in books.keys():
        msg_arr = msgs[sid]
        book_arr = books[sid]
        junction = cond_lens[sid]

        aggr_mask = msg_arr[:, 2].astype(int) == AGGRESSIVE_ORDER_ID
        aggr_indices = np.where(aggr_mask)[0]
        
        if len(aggr_indices) < 2:
            continue

        sizes = msg_arr[aggr_indices, 3].astype(float)
        prices = msg_arr[aggr_indices, 4].astype(float)

        first_idx = aggr_indices[0]
        ref_price = (book_arr[first_idx, 0] + book_arr[first_idx, 2]) / 2

        if ref_price <= 0:
            continue

        Q_cum = np.cumsum(sizes)
        notional_cum = np.cumsum(sizes * prices)
        vwap = notional_cum / np.maximum(Q_cum, eps)
        impact = np.abs(vwap - ref_price) / ref_price

        gen_msgs = msg_arr[junction:]
        exec_mask = gen_msgs[:, 1].astype(int) == 4
        V_exp = np.sum(gen_msgs[exec_mask, 3].astype(float)) if np.any(exec_mask) else 1.0
        V_exp = max(V_exp, eps)

        midprices = (book_arr[junction:, 0] + book_arr[junction:, 2]) / 2
        H, L = np.max(midprices), np.min(midprices)
        
        if H > L and L > 0:
            eta = np.log(H / L) / 0.8325546
            alphas.append(np.log(max(eta, eps)))

        valid_mask = impact > eps
        if np.sum(valid_mask) < 2:
            continue

        x_vals = np.log(Q_cum[valid_mask] / V_exp)
        y_vals = np.log(impact[valid_mask])
        iterations = np.arange(1, len(sizes) + 1)[valid_mask]
        
        for x, y, it in zip(x_vals, y_vals, iterations):
            points.append((x, y, int(it)))

    if not points or not alphas:
        print("No valid data")
        return

    # Convert to arrays
    X = np.array([p[0] for p in points])
    Y = np.array([p[1] for p in points])
    iterations = np.array([p[2] for p in points])
    
    alpha_global = np.mean(alphas)
    max_iter = int(iterations.max())
    
    # Show distribution of iterations
    print("=== Distribution of iterations ===")
    for i in range(1, max_iter + 1):
        cnt = np.sum(iterations == i)
        print(f"  iter={i}: {cnt} points")
    
    # Compute beta for each starting point a
    a_values = np.arange(1, max_iter + 1)
    betas = np.full_like(a_values, np.nan, dtype=float)
    n_points_used = np.zeros_like(a_values, dtype=int)
    
    print("\n=== Beta at each starting point ===")
    for idx, a in enumerate(a_values):
        mask = iterations >= a
        n_pts = np.sum(mask)
        
        if n_pts < 2:
            print(f"  a={a}: skip (only {n_pts} points)")
            continue
        
        X_a = X[mask]
        Y_a = Y[mask]
        y_adj = Y_a - alpha_global
        
        valid = np.isfinite(X_a) & np.isfinite(y_adj) & (X_a != 0)
        if np.sum(valid) < 2:
            continue
            
        betas[idx] = float(np.dot(X_a[valid], y_adj[valid]) / np.dot(X_a[valid], X_a[valid]))
        n_points_used[idx] = np.sum(valid)
        
        # Distance from theory
        dist = abs(betas[idx] - 0.5)
        print(f"  a={a}: β={betas[idx]:.4f}, |β-0.5|={dist:.4f}, points={n_points_used[idx]}")
    
    # Plot with secondary y-axis for number of points
    from plotly.subplots import make_subplots
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Beta evolution line
    hover_text = [f"a={a}<br>β={b:.4f}<br>|β-0.5|={abs(b-0.5):.4f}<br>points={n}" 
                  for a, b, n in zip(a_values, betas, n_points_used)]
    
    fig.add_trace(go.Scatter(
        x=a_values, y=betas,
        mode='lines+markers',
        marker=dict(size=10, color='blue'),
        line=dict(color='blue', width=2),
        text=hover_text, hoverinfo='text',
        name='β(a)'
    ), secondary_y=False)
    
    # Number of points (bar)
    fig.add_trace(go.Bar(
        x=a_values, y=n_points_used,
        marker=dict(color='rgba(200,200,200,0.5)'),
        name='N points',
        hoverinfo='skip'
    ), secondary_y=True)
    
    # Theory line β = 0.5
    fig.add_hline(y=0.5, line_dash="dash", line_color="green", line_width=2,
                  annotation_text="Theory β=0.5", annotation_position="right")
    
    fig.update_layout(
        title='Beta Evolution: β(a) — beta computed from iteration a onwards',
        xaxis_title='Starting iteration (a)',
        width=900, height=500,
        template='plotly_white',
        barmode='overlay'
    )
    fig.update_yaxes(title_text="β (OLS)", secondary_y=False)
    fig.update_yaxes(title_text="N points", secondary_y=True, showgrid=False)
    
    fig.show()

compute_beta_evolution(gen_books, gen_msgs, cond_lens)

No valid data
